# Drive Me Crazy — 1. Dataset analysis

Four public traffic datasets, the same ones the PDFormer paper (Jiang et al., AAAI 2023,
[arXiv 2301.07945](https://arxiv.org/abs/2301.07945)) is evaluated on:

| dataset | what | nodes | interval | period |
|---|---|---|---|---|
| PeMS04 | highway loop-detector **flow** (vehicles / 5 min), San Francisco Bay Area | 307 sensors | 5 min | Jan–Feb 2018 |
| PeMS07 | same, Los Angeles | 883 sensors | 5 min | May–Aug 2017 |
| PeMS08 | same, San Bernardino | 170 sensors | 5 min | Jul–Aug 2016 |
| NYCTaxi | taxi **inflow / outflow** per grid cell, Manhattan | 15 × 5 = 75 cells | 30 min | 2014 |

Every dataset is a 3-D array `(time, node, channel)` plus a graph telling which nodes are neighbours
(road links for PeMS, the 8 surrounding cells for the taxi grid). Run `uv run get_data.py` once to
download them (~160 MB, never committed).

This notebook answers three questions: what does the data look like, how clean is it, and — the
question behind this whole project — **is there a measurable propagation delay between neighbouring
sensors** that a forecasting model could exploit?

In [ ]:
import os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

DATA, RESULTS = Path("data"), Path("results")
RESULTS.mkdir(exist_ok=True)

# Per-dataset facts (paper: PDFormer, Jiang et al., AAAI 2023, Table 1) and the forecasting setup PDFormer uses.
# min_true: readings below it are treated as missing when scoring (0 = sensor gap; NYC cells with < 10 trips).
DATASETS = {
    "PEMS04":  dict(step_min=5,  start="2018-01-01", channels=["flow"],              min_true=1,  T_in=12, T_out=12, split=(0.6, 0.2, 0.2)),
    "PEMS08":  dict(step_min=5,  start="2016-07-01", channels=["flow"],              min_true=1,  T_in=12, T_out=12, split=(0.6, 0.2, 0.2)),
    "PEMS07":  dict(step_min=5,  start="2017-05-01", channels=["flow"],              min_true=1,  T_in=12, T_out=12, split=(0.6, 0.2, 0.2)),
    "NYCTaxi": dict(step_min=30, start="2014-01-01", channels=["inflow", "outflow"], min_true=10, T_in=6,  T_out=1,  split=(0.7, 0.1, 0.2)),
}
# DMC_DATASETS="PEMS08" restricts a run (used by the Docker smoke test); default = all four.
SELECTED = [d for d in os.environ.get("DMC_DATASETS", ",".join(DATASETS)).split(",") if d]


def grid_adjacency(rows, cols):
    """8-neighbour adjacency of a rows x cols grid; cell index = row * cols + col."""
    r, c = np.divmod(np.arange(rows * cols), cols)
    adj = (np.abs(r[:, None] - r[None]) <= 1) & (np.abs(c[:, None] - c[None]) <= 1)
    np.fill_diagonal(adj, False)
    return adj.astype(np.float32)


def load_dataset(name):
    """X (T, N, C) float32 targets, adj (N, N) binary symmetric, times DatetimeIndex of length T."""
    cfg = DATASETS[name]
    if name.startswith("PEMS"):
        X = np.load(DATA / f"{name}.npz")["data"][..., :1].astype(np.float32)  # channel 0 = traffic flow
        adj = np.load(DATA / f"adj_{name}.npy").astype(np.float32)
    else:
        df = pd.read_csv(DATA / "NYCTaxi" / "NYCTaxi.grid")
        rows, cols, T = df.row_id.max() + 1, df.column_id.max() + 1, df.time.nunique()
        assert df.dyna_id.is_monotonic_increasing  # cell-major file: every timestep of cell 0, then cell 1, ...
        X = df[cfg["channels"]].to_numpy(np.float32).reshape(rows * cols, T, len(cfg["channels"])).transpose(1, 0, 2)
        adj = grid_adjacency(rows, cols)
    times = pd.date_range(cfg["start"], periods=len(X), freq=f"{cfg['step_min']}min")
    return np.ascontiguousarray(X), adj, times


def time_features(times, step_min):
    """Time-of-day slot index and day-of-week for every timestep."""
    tod = ((times.hour * 60 + times.minute) // step_min).to_numpy()
    return tod, times.dayofweek.to_numpy()


def windows(X, T_in, T_out):
    """Zero-copy sliding windows of length T_in + T_out: (n, T_in + T_out, N, C) view. Index it to copy a batch."""
    return np.moveaxis(sliding_window_view(X, T_in + T_out, axis=0), -1, 1)


def split_indices(n_windows, split):
    """Chronological train / val / test window indices."""
    a, b = int(n_windows * split[0]), int(n_windows * (split[0] + split[1]))
    return np.arange(a), np.arange(a, b), np.arange(b, n_windows)


def masked_metrics(pred, true, min_true):
    """MAE / RMSE / MAPE over entries whose ground truth is >= min_true."""
    m = true >= min_true
    err = (pred - true)[m]
    return dict(MAE=np.abs(err).mean(), RMSE=np.sqrt((err ** 2).mean()), MAPE=100 * (np.abs(err) / true[m]).mean())


def metrics_by_horizon(pred, true, min_true):
    """pred / true (n, T_out, N, C) -> one row per horizon plus an 'avg' row (mean of the per-horizon metrics)."""
    rows = [dict(horizon=str(h + 1), **masked_metrics(pred[:, h], true[:, h], min_true)) for h in range(pred.shape[1])]
    df = pd.DataFrame(rows)
    avg = df.drop(columns="horizon").mean().to_dict()
    return pd.concat([df, pd.DataFrame([dict(horizon="avg", **avg)])], ignore_index=True)


def sample_nodes(adj, X):
    """Three nodes to plot: the best-connected sensor, the busiest, and the quietest (non-dead)."""
    mean_flow = X[..., 0].mean(0)
    alive = np.where(mean_flow > np.percentile(mean_flow, 5))[0]
    return sorted({int(adj.sum(0).argmax()), int(mean_flow.argmax()), int(alive[mean_flow[alive].argmin()])})


def sample_frame(times, test_idx, T_in, T_out, nodes, true, preds, n_days, step_min, model_names):
    """Long table (time, node, actual, <model>_h1, <model>_hLast) over the first n_days of the test period."""
    steps = n_days * 24 * 60 // step_min
    rows = []
    for k in range(min(steps, len(test_idx))):
        w = test_idx[k]
        t1, tL = times[w + T_in], times[w + T_in + T_out - 1]
        for n in nodes:
            rec = dict(time_h1=t1, time_hL=tL, node=n, actual_h1=true[k, 0, n, 0], actual_hL=true[k, -1, n, 0])
            for name, p in zip(model_names, preds):
                rec[f"{name}_h1"], rec[f"{name}_hL"] = p[k, 0, n, 0], p[k, -1, n, 0]
            rows.append(rec)
    return pd.DataFrame(rows)

## 1. Shape, coverage, missing readings

In [ ]:
data = {name: load_dataset(name) for name in SELECTED}
rows = []
for name, (X, adj, times) in data.items():
    cfg = DATASETS[name]
    deg = adj.sum(1)
    rows.append(dict(
        dataset=name, nodes=X.shape[1], timesteps=len(X), channels="+".join(cfg["channels"]),
        interval_min=cfg["step_min"], start=times[0].date(), end=times[-1].date(), days=round(len(X) * cfg["step_min"] / 1440),
        edges=int(adj.sum() // 2), mean_degree=round(float(deg.mean()), 2), isolated_nodes=int((deg == 0).sum()),
        zero_readings_pct=round(float((X[..., 0] == 0).mean() * 100), 2),
        mean=round(float(X[..., 0].mean()), 1), p99=round(float(np.percentile(X[..., 0], 99)), 1), max=float(X[..., 0].max()),
    ))
summary = pd.DataFrame(rows).set_index("dataset")
summary.to_csv(RESULTS / "dataset_summary.csv")
summary

Observations:

- No NaNs anywhere: gaps are encoded as **zeros** (0.3–1.6 % of PeMS readings). A zero flow at 8 am on a
  highway is a dead sensor, not an empty road, so every metric in this project ignores ground-truth zeros
  (and NYC cells with fewer than 10 trips, as the paper does).
- The graphs are sparse: 2–3 neighbours per sensor on the highways, up to 8 on the taxi grid. Whatever
  a model learns about "space" has to come through very few links — a long-range mechanism (attention)
  rather than local averaging (GCN) is the obvious candidate.
- Scales differ by an order of magnitude between sensors and datasets, so models are trained on
  standardised values and scored on the original scale.

## 2. Daily and weekly rhythm

In [ ]:
fig, axes = plt.subplots(2, len(data), figsize=(5 * len(data), 7), squeeze=False)
for j, (name, (X, adj, times)) in enumerate(data.items()):
    cfg = DATASETS[name]
    tod, dow = time_features(times, cfg["step_min"])
    spd = 1440 // cfg["step_min"]
    flow = X[..., 0].mean(1)  # network-wide mean per timestep
    prof = pd.DataFrame({"tod": tod, "weekend": dow >= 5, "flow": flow}).groupby(["weekend", "tod"]).flow.mean().unstack(0)
    hours = np.arange(spd) * cfg["step_min"] / 60
    axes[0, j].plot(hours, prof[False], label="weekday"); axes[0, j].plot(hours, prof[True], label="weekend")
    axes[0, j].set_title(f"{name}: mean daily profile ({cfg['channels'][0]})"); axes[0, j].set_xlabel("hour"); axes[0, j].legend()
    node = sample_nodes(adj, X)[0]
    week = slice(7 * spd, 14 * spd)
    axes[1, j].plot(times[week], X[week, node, 0], lw=0.7)
    axes[1, j].set_title(f"{name}: one week, node {node}"); axes[1, j].tick_params(axis="x", rotation=30)
plt.tight_layout()

The strong, repeatable daily shape (two rush-hour peaks on the highways, a nightlife bump for taxis) is
what a *Historical Average* baseline will exploit. The week-of-one-sensor plots show what it cannot: the
day-to-day variation around that shape — incidents, weather, holidays — is large, and that is the part a
model has to forecast from the last hour of observations.

## 3. Value distributions and graph degree

In [ ]:
fig, axes = plt.subplots(2, len(data), figsize=(5 * len(data), 6), squeeze=False)
for j, (name, (X, adj, times)) in enumerate(data.items()):
    axes[0, j].hist(X[..., 0].ravel(), bins=80, log=True, color="C0"); axes[0, j].set_title(f"{name}: {DATASETS[name]['channels'][0]} distribution (log count)")
    axes[1, j].hist(adj.sum(1), bins=np.arange(0, adj.sum(1).max() + 2) - 0.5, color="C1"); axes[1, j].set_title(f"{name}: node degree")
plt.tight_layout()

## 4. Is there a propagation delay?

If traffic really "travels" along the road, a sensor's flow should correlate best with its downstream
neighbour's flow **a few minutes later**, not at the same instant. Two neighbouring sensors share the same
rush hours, so their raw series correlate at ~0.9 at *every* lag and the delay is invisible; we therefore
first subtract each sensor's mean daily profile and correlate the **residuals** — the part of the traffic
that is unusual today, which is what actually propagates. For every edge `(i, j)` we compute the
correlation between `r_i(t)` and `r_j(t + lag)` for lags of −12…+12 steps (±1 h for PeMS, ±6 h for
NYCTaxi) over the first two weeks, and record the lag with the highest correlation.

In [ ]:
MAX_LAG = 12
lag_rows, curves = [], {}
for name, (X, adj, times) in data.items():
    cfg = DATASETS[name]
    spd = 1440 // cfg["step_min"]
    horizon = 14 * spd
    x = X[:horizon, :, 0]
    x = x - x.reshape(14, spd, -1).mean(0)[np.tile(np.arange(spd), 14)]   # residual after the mean daily profile
    x = (x - x.mean(0)) / (x.std(0) + 1e-6)
    src, dst = np.where(np.triu(adj) > 0)
    corr = np.zeros((2 * MAX_LAG + 1, len(src)))
    for k, lag in enumerate(range(-MAX_LAG, MAX_LAG + 1)):
        a = x[max(0, -lag):len(x) - max(0, lag), src]
        b = x[max(0, lag):len(x) - max(0, -lag), dst]
        corr[k] = (a * b).mean(0)
    best = corr.argmax(0) - MAX_LAG
    curves[name] = corr.mean(1)
    lag_rows += [dict(dataset=name, best_lag=int(b), abs_lag_min=int(abs(b)) * cfg["step_min"], corr_at_0=float(corr[MAX_LAG, e]), corr_best=float(corr[best[e] + MAX_LAG, e])) for e, b in enumerate(best)]
lags = pd.DataFrame(lag_rows)
lags.to_csv(RESULTS / "lag_analysis.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, c in curves.items():
    axes[0].plot(range(-MAX_LAG, MAX_LAG + 1), c, marker="o", label=name)
axes[0].set_xlabel("lag (steps): r_j(t + lag) vs r_i(t)"); axes[0].set_ylabel("mean residual correlation over edges"); axes[0].legend(); axes[0].set_title("Cross-correlation of residuals along graph edges")
share = pd.crosstab(lags.best_lag.abs().clip(upper=4), lags.dataset, normalize="columns")[list(curves)]
share.index = [str(i) if i < 4 else "4+" for i in share.index]
share.plot.bar(ax=axes[1], width=0.8, rot=0)
axes[1].set_xlabel("|best lag| (steps)"); axes[1].set_ylabel("share of edges"); axes[1].set_title("Where each edge's residual correlation peaks")
plt.tight_layout()

delay_share = lags.groupby("dataset").apply(lambda d: pd.Series({
    "edges": len(d), "share_peak_off_zero": (d.best_lag != 0).mean(),
    "median_abs_lag_min": d.abs_lag_min.median(), "corr_gain_from_lag": (d.corr_best - d.corr_at_0).mean()}), include_groups=False)
delay_share.to_csv(RESULTS / "lag_summary.csv")
delay_share.round(3)

Reading the result (numbers in the table above):

- **PeMS04 and PeMS08**: 83–87 % of the edges peak at a *non-zero* lag, most often one 5-minute step
  (63 % of PeMS04 edges, 43 % of PeMS08), with a tail of edges whose peak sits 20 minutes or more out.
  What is unusual at a sensor right now shows up at its neighbour minutes later — the "propagation delay"
  the PDFormer paper builds a module for, measured directly.
- **PeMS07** is the exception among the highways: 82 % of edges peak at lag 0 and the lag-0 residual
  correlation (0.8) is the sharpest of all. Its 883 detectors sit densely along the Los Angeles freeways,
  so neighbours are close enough that the delay is shorter than the 5-minute sampling.
- **NYCTaxi**: 57 % at lag 0, 27 % at one step — but one step is 30 minutes, and the triangular curve
  shows that a cell's anomaly has decayed within a couple of hours. Whatever delay exists between
  adjacent cells is much shorter than the bin; the resolution hides it.
- The correlation *gain* from choosing the best lag is small everywhere (0.005–0.02): the delay is real
  but second-order next to the instantaneous correlation. A delay-aware model should therefore help most
  on PeMS04/08, marginally on PeMS07, and least on NYCTaxi. `drive_me_crazy_pdformer.ipynb` tests
  exactly that with an ablation.